# Prompt Engineering

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы: 
* https://www.promptingguide.ai/techniques/fewshot

## Задачи для совместного разбора

<p class="task" id="1"></p>

1\. Обсудите основные техники промптинга для современных LLM.

## Задачи для самостоятельного решения

In [28]:
import os
import random
from typing import List, Dict, Any, Optional, Literal
from collections import Counter

from IPython.display import display, Markdown
import pandas as pd
import torch as th
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

os.makedirs("data", exist_ok=True)

vkr_path = "data/prikaz-po-VKR-bak-i-mag_izmeneniya.txt"
graph_path = "data/graph.csv"

<p class="task" id="1"></p>

1\. В этой задаче ваша цель — заставить модель рассуждать как конкретный персонаж, не описывая его характер словами, а показав примеры его логики. Для этого нужно использовать технику few-shot промптинг  для переноса когнитивного стиля (способа мышления) эксперта.

Дан пример выступления профессора, который всегда ищет исторические аналогии, отрицает современные "модные" термины и рассматривает любую проблему как системную ошибку.

Спросите модель без каких-либо дополнительных системных промптов: "Стоит ли запретить смартфоны в школах?". Выведите ответ на экран.

Составьте промпт, который содержит три примера диалогов с профессором, заставив модель извлечь паттерн мышления только из примеров. В конце промпта добавьте целевой вопрос: "Стоит ли запретить смартфоны в школах?". Выведите результат на экран

Сравните ответ zero-shot и few-shot. Прокомментируйте результат

- [x] Проверено на семинаре

In [29]:
examples = [
    {
        "question": "Полезна ли геймификация в школах?",
        "answer": "Прежде чем говорить о пользе, нужно определить цель. Если цель — дрессировка рефлексов, как у собаки Павлова, то да, полезно. Но если мы смотрим на образование как на эволюционный процесс, то внешняя стимуляция атрофирует внутреннюю волю. Мы подменяем смыслы фантиками."
    },
    {
        "question": "Нужно ли учить детей Soft Skills?",
        "answer": "Сам термин 'мягкие навыки' — это корпоративный новояз. Античные греки называли это этикой и риторикой. Пытаясь выделить это в отдельную дисциплину, мы совершаем ошибку редукционизма. Нельзя научить эмпатии по учебнику, как нельзя научиться плавать, изучая формулу воды. Это свойство системы, а не навык."
    },
    {
        "question": "Заменят ли онлайн-курсы университеты?",
        "answer": "Это всё равно что спросить, заменят ли консервы свежую охоту. Информационно — да, биологически — нет. Университет — это не набор лекций, это среда обитания, формирующая мышление через социальное трение. Убирая стены, мы убираем сопротивление, необходимое для роста интеллектуальных мышц."
    }
]

In [30]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

target_question = "Стоит ли запретить смартфоны в школах?"

zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])
zero_shot_chain = zero_shot_prompt | llm
zero_shot_res = zero_shot_chain.invoke({"question": target_question})
display(Markdown(zero_shot_res.content))

Вопрос о запрете смартфонов в школах вызывает много споров и обсуждений. Рассмотрим некоторые аргументы "за" и "против":

### Аргументы "за" запрет:
1. **Сосредоточенность на учебе**: Смартфоны могут отвлекать учеников от занятий, что негативно сказывается на их успеваемости.
2. **Социальные взаимодействия**: Удаление смартфонов может способствовать более активному общению между учениками и укреплению социальных связей.
3. **Кибербуллинг**: Смартфоны могут быть инструментом для издевательств и травли, и их отсутствие может уменьшить такие случаи.
4. **Проблемы с психическим здоровьем**: Чрезмерное использование смартфонов может привести к проблемам с вниманием, тревожностью и депрессией.

### Аргументы "против" запрета:
1. **Образовательные возможности**: Смартфоны могут быть полезными инструментами для обучения, предоставляя доступ к информации, образовательным приложениям и онлайн-ресурсам.
2. **Подготовка к жизни**: Умение пользоваться технологиями является важным навыком в современном мире, и запрет может лишить учеников этой возможности.
3. **Безопасность**: Смартфоны могут быть важным средством связи в экстренных ситуациях.
4. **Индивидуальный подход**: Не все ученики используют смартфоны во вред; многие могут ответственно подходить к их использованию.

### Вывод
Запрет смартфонов в школах может иметь как положительные, так и отрицательные последствия. Возможно, более уместным решением будет регулирование их использования, например, разрешение на использование смартфонов только в определенное время или для выполнения конкретных задач. Каждая школа может рассмотреть свои особенности и потребности, прежде чем принимать окончательное решение.

In [31]:
few_shot_messages = []
for ex in examples:
    few_shot_messages.append(HumanMessage(content=ex["question"]))
    few_shot_messages.append(AIMessage(content=ex["answer"]))

few_shot_messages.append(HumanMessage(content=target_question))

few_shot_res = llm.invoke(few_shot_messages)
display(Markdown(few_shot_res.content))

Запретить смартфоны в школах — это подход, который может иметь как свои плюсы, так и минусы. С одной стороны, смартфоны могут отвлекать учеников от учебного процесса, способствуя снижению концентрации и ухудшению успеваемости. С другой стороны, они могут служить полезным инструментом для обучения, предоставляя доступ к информации и образовательным приложениям.

Вместо полного запрета, возможно, стоит рассмотреть возможность регулирования использования смартфонов. Например, можно установить определенные правила о том, когда и как можно использовать устройства, а также внедрить программы по обучению цифровой грамотности и ответственности.

Важно также учитывать, что современные технологии становятся неотъемлемой частью жизни, и учить детей правильно использовать их — это тоже важный аспект образования.

<p class="task" id="2"></p>

2\. Исследуйте способность модели удерживать динамическое состояние объекта при жестком ограничении длины ответа. Для этой задачи не требуется использовать самые современные продвинутые reasoning‑модели, достаточно модели уровня GPT‑4o‑mini.

Используйте следующий текст задачи об игре в наперстки:
```
Есть 3 стакана: A, B, C. Мячик лежит под стаканом B.
Происходит серия перестановок:
Поменяли местами A и B.
Поменяли местами B и C.
Поменяли местами A и C.
Поменяли местами A и B.
Поменяли местами B и C.
Поменяли местами A и C.
Где сейчас мяч?
```

Сформулируйте системный промпт, который строго запрещает рассуждения в ответе. Ответ должен содержать только итоговую букву без пояснений.

Затем сформулируйте промпт, который явно требует отслеживать положение мяча после каждого шага. Сравните ответы. Проверьте правильность цепочки рассуждений во втором случае. 

- [x] Проверено на семинаре

In [32]:
shell_game_text = """
Есть 3 стакана: A, B, C. Мячик лежит под стаканом B.
Происходит серия перестановок:
Поменяли местами A и B.
Поменяли местами B и C.
Поменяли местами A и C.
Поменяли местами A и B.
Поменяли местами B и C.
Поменяли местами A и C.
Где сейчас мяч?
"""

no_reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — строго ограниченный исполнитель. Твоя задача — выдать ответ на задачу. "
               "Тебе СТРОГО запрещено рассуждать, писать цепочки мыслей или давать пояснения. "
               "Ответом должна быть строго одна заглавная буква (A, B или C)."),
    ("human", "{text}")
])

no_reasoning_chain = no_reasoning_prompt | llm
no_res = no_reasoning_chain.invoke({"text": shell_game_text})

display(Markdown(f"Ответ модели: '{no_res.content}'"))
display(Markdown(f"Длина ответа (в символах): {len(no_res.content)}"))

Ответ модели: 'C'

Длина ответа (в символах): 1

In [33]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — внимательный аналитик. Тебе необходимо решить задачу, "
               "обязательно отслеживая и записывая положение мяча ПОСЛЕ КАЖДОГО шага перестановок. "
               "В самом конце укажи итоговую букву стакана."),
    ("human", "{text}")
])

cot_chain = cot_prompt | llm
cot_res = cot_chain.invoke({"text": shell_game_text})

display(Markdown(f"Ответ модели: '{cot_res.content}'"))

Ответ модели: 'Давайте отслеживать положение мяча под стаканами после каждой перестановки.

1. Начальное положение: 
   - A: пусто
   - B: мяч
   - C: пусто

2. Перестановка A и B:
   - A: мяч
   - B: пусто
   - C: пусто

3. Перестановка B и C:
   - A: мяч
   - B: пусто
   - C: пусто (стакан C остается пустым)

4. Перестановка A и C:
   - A: пусто
   - B: пусто
   - C: мяч

5. Перестановка A и B:
   - A: пусто
   - B: пусто
   - C: мяч (стакан A остается пустым)

6. Перестановка B и C:
   - A: пусто
   - B: мяч
   - C: пусто

7. Перестановка A и C:
   - A: мяч
   - B: пусто
   - C: пусто

Итак, после всех перестановок мяч находится под стаканом A.'

обе модели дали неправильный ответ)


<p class="task" id="3"></p>

3\. Повторите решение задачи 2, используя системный промпт, запрещающий рассуждения, но теперь укажите высокую температуру при генерации ответа для обеспечения разнообразия. Выполните несколько независимых запусков модели с одним и тем же промптом. Соберите все ответы и примените методику голосования большинством для выбора финального ответа. Прокомментируйте, удается ли таким образом преодолеть ограничения промпта.

- [x] Проверено на семинаре

In [34]:
high_temp_llm = ChatOpenAI(model="gpt-4o-mini", temperature=1.5)

no_reasoning_high_temp_chain = no_reasoning_prompt | high_temp_llm

num_runs = 17
responses = []

print(f"Запуск {num_runs} независимых генераций без рассуждений...")
for i in range(num_runs):
    res = no_reasoning_high_temp_chain.invoke({"text": shell_game_text})
    clean_res = res.content.strip().upper()
    if len(clean_res) > 0:
        clean_res = clean_res[0]
    responses.append(clean_res)
    print(f"Запуск {i+1}: '{clean_res}'")

votes = Counter(responses)
final_answer = votes.most_common(1)[0][0]


for ans, count in votes.items():
    print(f"Ответ '{ans}': {count} голосов ({count/num_runs*100:.1f}%)")

print(f"\nИтоговый ансамблированный ответ: {final_answer} (Правильный: B)")

Запуск 17 независимых генераций без рассуждений...
Запуск 1: 'C'
Запуск 2: 'C'
Запуск 3: 'A'
Запуск 4: 'C'
Запуск 5: 'C'
Запуск 6: 'C'
Запуск 7: 'C'
Запуск 8: 'A'
Запуск 9: 'A'
Запуск 10: 'C'
Запуск 11: 'C'
Запуск 12: 'A'
Запуск 13: 'A'
Запуск 14: 'A'
Запуск 15: 'A'
Запуск 16: 'C'
Запуск 17: 'C'
Ответ 'C': 10 голосов (58.8%)
Ответ 'A': 7 голосов (41.2%)

Итоговый ансамблированный ответ: C (Правильный: B)


In [35]:
from IPython.display import Image

display(Image(url="https://i.makeagif.com/media/6-03-2016/EGVgYj.gif"))
display(Image(url="https://i.makeagif.com/media/10-09-2015/9me5a5.gif"))

<p class="task" id="4"></p>

4\. Уточнив промпт, доработайте созданного ранее RAG-агента для ответов на вопросы по документу о ВКР таким образом, чтобы он соблюдал следующие правила:
- всегда указывать номера пунктов документа, на которых основан ответ.
- если информация прямо есть в документе, отвечать одним предложением.
- если информации нет, делать логический вывод, используя few-shot примеры reasoning в виде "пункт 1.1 -> вывод 1 -> вывод Y -> итог"

Протестируйте агента на на 2 вопросах:
- ответ на который есть в документе
- ответа нет, но можно вывести логически

- [x] Проверено на семинаре

In [36]:
loader = TextLoader(vkr_path, encoding="utf-8")
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100).split_documents(loader.load())

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True}
)

# Векторное хранилище
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
rag_system_prompt = """Ты — юридически точный RAG-ассистент учебного отдела. Твоя задача — отвечать на вопросы студентов строго на основе предоставленного Текста Документа.

ПРАВИЛА ОТВЕТА:
1. Обязательно указывай номер пункта документа, на котором основан твой ответ.
2. Если ответ прямо содержится в документе, дай его строго в ОДНО ПРЕДЛОЖЕНИЕ.
3. Если прямого ответа на вопрос в документе нет, но его можно логически вывести, ты должен построить цепочку логических рассуждений строго в формате:
   "пункт X.Y -> вывод 1 -> вывод 2 -> итог"
   
ПРИМЕРЫ ЛОГИЧЕСКОГО ВЫВОДА (Few-Shot):
Вопрос: "Могу ли я изменить тему ВКР за 15 дней до ГИА?"
Ответ: "пункт 2.9 -> для изменения темы требуется не менее 1 месяца (30 дней) -> 15 дней меньше установленного срока в 1 месяц -> итог: изменить тему за 15 дней до ГИА невозможно."

Вопрос: "Я учусь в магистратуре, сколько минут мне готовить доклад?"
Ответ: "пункт 5.12 -> для программ магистратуры на доклад отводится не более 15 минут -> итог: вам необходимо подготовить доклад длительностью до 15 минут."
"""

def run_rag_agent(question: str) -> str:
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    messages = [
        SystemMessage(content=rag_system_prompt),
        HumanMessage(content=f"Текст Документа:\n{context}\n\nВопрос студента: {question}")
    ]
    
    eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    response = eval_llm.invoke(messages)
    
    return response.content

In [38]:
q_direct = "Какое максимальное время отводится на доклад бакалавру?"
q_logical = "Студент Глеб хочет изменить тему своей бакалаврской работы за 12 дней до начала ГИA. Разрешит ли ему учебный офис это сделать?"

display(Markdown(f"`Вопрос`: {q_direct}"))
display(Markdown(f"`Ответ`: {run_rag_agent(q_direct)}"))


display(Markdown(f"`Вопрос`: {q_logical}"))
display(Markdown(f"`Ответ`: {run_rag_agent(q_logical)}"))

`Вопрос`: Какое максимальное время отводится на доклад бакалавру?

`Ответ`: "пункт 4.6 -> для обучающихся по программе бакалавриата на доклад отводится 7-10 минут -> итог: максимальное время на доклад бакалавру составляет 10 минут."

`Вопрос`: Студент Глеб хочет изменить тему своей бакалаврской работы за 12 дней до начала ГИA. Разрешит ли ему учебный офис это сделать?

`Ответ`: "пункт 2.9 -> изменение темы ВКР возможно не позднее, чем за 1 месяц (30 дней) -> 12 дней меньше установленного срока в 1 месяц -> итог: изменить тему за 12 дней до ГИА невозможно."

In [42]:
display(Markdown(f"`Ответ`: {run_rag_agent("Сколько лет декану факультета?")}"))

`Ответ`: В документе нет информации о возрасте декана факультета.

<p class="task" id="5"></p>

5\. Дан граф социальных связей в формате csv `graph.csv`. Каждый человек может быть другом, коллегой или руководителем другого. Требуется разработать промпт, который позволит отвечать на вопросы по графу, заданные в свободной форме.

Для этого добавьте в промпт фрагмент, содержащий данные из графа:

```
Граф социальных связей:
Алиса -> Борис (друг)
...
```

В промпте дополнительно уточните, как модель должна рассуждать (использовать информацию о графе) для поиска ответа на вопрос. В ответе модель должна демонстрировать путь рассуждений. Покажите работу на нескольких примерах вопросов.


- [x] Проверено на семинаре

In [39]:
graph_df = pd.read_csv("data/graph.csv", encoding="utf-8")

graph_triples = []
for _, row in graph_df.iterrows():
    graph_triples.append(f"{row['source']} -> {row['target']} ({row['relation']})")

graph_representation = "Граф социальных связей:\n" + "\n".join(graph_triples)

print(graph_representation)


Граф социальных связей:
Алиса -> Борис (friend)
Алиса -> Виктор (colleague)
Борис -> Диана (friend)
Виктор -> Диана (manager)
Диана -> Ева (friend)
Ева -> Алиса (colleague)
Виктор -> Фёдор (friend)
Фёдор -> Борис (colleague)


In [40]:
graph_system_prompt = """Ты — эксперт по теории графов и сетевому анализу. Твоя задача — отвечать на вопросы по предоставленному Графу Социальных Связей.

Для поиска ответа ты обязан выполнить пошаговый обход графа (логический вывод):
1. Найди в графе начальную вершину (человека), упомянутую в вопросе.
2. Выпиши все её исходящие и входящие связи, имеющие отношение к вопросу.
3. Проследуй по цепочке связей к целевой вершине.
4. Сделай вывод на основе найденного пути.

В ответе ОБЯЗАТЕЛЬНО продемонстрируй свой путь рассуждений по шагам.
"""

def query_graph(question: str) -> str:
    messages = [
        SystemMessage(content=graph_system_prompt),
        HumanMessage(content=f"{graph_representation}\n\nВопрос: {question}")
    ]
    
    graph_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    response = graph_llm.invoke(messages)
    return response.content

In [41]:
questions = [
    "У Дианы есть два пути получения информации от Алисы: один — сугубо дружеский, другой — профессиональный (через коллег и менеджеров). Опиши оба пути шаг за шагом.",
    "Виктор хочет познакомиться с Борисом. Через какого своего друга он может это сделать и кем этот общий знакомый приходится Борису?",
    "Существует ли замкнутый цикл, проходящий через Алису, в котором цепочка рукопожатий возвращается обратно к ней? Опиши этот путь."
]

for i, q in enumerate(questions, 1):
    display(Markdown(f"**ВОПРОС {i}**"))
    display(Markdown(f"`Вопрос`: {q}"))
    display(Markdown(f"`Ответ`: {query_graph(q)}"))

**ВОПРОС 1**

`Вопрос`: У Дианы есть два пути получения информации от Алисы: один — сугубо дружеский, другой — профессиональный (через коллег и менеджеров). Опиши оба пути шаг за шагом.

`Ответ`: Для ответа на вопрос о путях получения информации от Алисы к Диане, я выполню пошаговый обход графа для обоих типов связей: дружеской и профессиональной.

### Дружеский путь:

1. **Начальная вершина**: Алиса.
2. **Исходящие связи от Алисы**: 
   - Алиса -> Борис (friend)
   - Алиса -> Виктор (colleague)
3. **Следуем по дружеской связи**: 
   - Алиса -> Борис (friend).
4. **Исходящие связи от Бориса**: 
   - Борис -> Диана (friend).
5. **Следуем по дружеской связи**: 
   - Борис -> Диана (friend).

**Вывод для дружеского пути**: 
Алиса может передать информацию Диане через Бориса, используя дружескую связь: Алиса -> Борис -> Диана.

### Профессиональный путь:

1. **Начальная вершина**: Алиса.
2. **Исходящие связи от Алисы**: 
   - Алиса -> Борис (friend)
   - Алиса -> Виктор (colleague)
3. **Следуем по профессиональной связи**: 
   - Алиса -> Виктор (colleague).
4. **Исходящие связи от Виктора**: 
   - Виктор -> Диана (manager).
5. **Следуем по профессиональной связи**: 
   - Виктор -> Диана (manager).

**Вывод для профессионального пути**: 
Алиса может передать информацию Диане через Виктора, используя профессиональную связь: Алиса -> Виктор -> Диана.

### Итог:
Таким образом, у Дианы есть два пути получения информации от Алисы:
1. Дружеский путь: Алиса -> Борис -> Диана.
2. Профессиональный путь: Алиса -> Виктор -> Диана.

**ВОПРОС 2**

`Вопрос`: Виктор хочет познакомиться с Борисом. Через какого своего друга он может это сделать и кем этот общий знакомый приходится Борису?

`Ответ`: Давайте разберем вопрос по шагам.

1. **Начальная вершина**: Виктор. Мы ищем, через кого Виктор может познакомиться с Борисом.

2. **Исходящие и входящие связи Виктора**:
   - Исходящие связи:
     - Виктор -> Диана (manager)
     - Виктор -> Фёдор (friend)
   - Входящие связи:
     - Алиса -> Виктор (colleague)

3. **Целевая вершина**: Борис. Теперь мы проследим, как Виктор может добраться до Бориса через его связи.

   - Из исходящих связей Виктора у нас есть:
     - Виктор -> Диана
     - Виктор -> Фёдор

   - Начнем с Дианы:
     - Диана -> Борис (friend)
     - Это значит, что Виктор может познакомиться с Борисом через Диану.

   - Теперь проверим Фёдора:
     - Фёдор -> Борис (colleague)
     - Это также означает, что Виктор может познакомиться с Борисом через Фёдора.

4. **Вывод**: Виктор может познакомиться с Борисом через двух своих друзей: Диану и Фёдора. 

Теперь определим, кем эти общие знакомые приходятся Борису:
- Диана — друг Бориса.
- Фёдор — коллега Бориса.

Таким образом, ответ на вопрос: Виктор может познакомиться с Борисом через Диану или Фёдора. Диана — друг Бориса, а Фёдор — коллега Бориса.

**ВОПРОС 3**

`Вопрос`: Существует ли замкнутый цикл, проходящий через Алису, в котором цепочка рукопожатий возвращается обратно к ней? Опиши этот путь.

`Ответ`: Чтобы ответить на вопрос о существовании замкнутого цикла, проходящего через Алису, я выполню следующие шаги:

1. **Начальная вершина**: Начинаем с Алисы.

2. **Исходящие и входящие связи Алисы**:
   - Исходящие связи:
     - Алиса -> Борис (friend)
     - Алиса -> Виктор (colleague)
   - Входящие связи:
     - Ева -> Алиса (colleague)

3. **Проследуем по цепочке связей**:
   - Начнем с исходящей связи Алисы к Борису:
     - Алиса -> Борис
     - Далее, у Бориса есть связь к Диане:
       - Борис -> Диана
       - У Дианы есть связь к Еве:
         - Диана -> Ева
         - У Евы есть связь обратно к Алисе:
           - Ева -> Алиса
     - Таким образом, мы получили путь: Алиса -> Борис -> Диана -> Ева -> Алиса. Это замкнутый цикл.

4. **Вывод**: Да, существует замкнутый цикл, проходящий через Алису, который выглядит следующим образом: Алиса -> Борис -> Диана -> Ева -> Алиса. Цепочка рукопожатий возвращается обратно к Алисе.